[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/03_finetune.ipynb)

# Notebook 3 — Fine-Tuning with Unsloth

LoRA fine-tune LFM2.5-1.2B-Thinking on StepGame spatial reasoning data.

In [ ]:
import os, sys
REPO = '/content/spatialft.github.io'
if not os.path.exists(REPO):
    !git clone https://github.com/spatialft/spatialft.github.io.git {REPO}
os.chdir(f'{REPO}/notebooks')
if REPO not in sys.path:
    sys.path.insert(0, REPO)


In [ ]:
# Colab: install first
# !pip install -r ../requirements.txt

In [ ]:
import json
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

from src.dataset import SYSTEM_PROMPT

In [ ]:
MODEL_ID       = 'LiquidAI/LFM2.5-1.2B-Thinking'
MAX_SEQ_LENGTH = 512
LORA_RANK      = 16
OUTPUT_DIR     = '../results/finetuned/checkpoint'

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

In [ ]:
with open('../data/processed/train_formatted.json') as f:
    train_data = json.load(f)

dataset = Dataset.from_list(train_data)
print(f'Training on {len(dataset)} examples')
print(dataset[0]['full_text'][:300])

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field='full_text',
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=20,
        output_dir=OUTPUT_DIR,
        save_strategy='epoch',
        optim='adamw_8bit',
        seed=42,
    ),
)

trainer.train()

In [ ]:
# Save LoRA adapter
model.save_pretrained('../results/finetuned/lora_adapter')
tokenizer.save_pretrained('../results/finetuned/lora_adapter')
print('Adapter saved.')